# Student Mental Health Classification


## 1. Import the libraries

In [ ]:
# %pip install pandas matplotlib scikit-learn joblib

import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
)

## 2. Load the data
The target is `Mental_Health_Risk`. `Student_ID` is just an identifier.

In [ ]:
df = pd.read_csv("data/student_mental_health.csv")
display(df.head())
print("Shape:", df.shape)
print("Classes:", df["Mental_Health_Risk"].unique())
display(df.isna().sum())

## 3. Count and plot the classes


In [ ]:
classes = ["Low", "Medium", "High"]
counts = df["Mental_Health_Risk"].value_counts().reindex(classes)
display(pd.DataFrame({"Count": counts, "Percent": counts / len(df) * 100}))

counts.plot.bar(title="Class distribution", rot=0)
plt.ylabel("Number of students")
plt.show()

## 4. Split: 60% train, 20% validation, 20% test

In [ ]:
X = df.drop(columns=["Student_ID", "Mental_Health_Risk"])
y = df["Mental_Health_Risk"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42
)

print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))
display(
    pd.DataFrame(
        {
            "Train": y_train.value_counts(),
            "Validation": y_val.value_counts(),
            "Test": y_test.value_counts(),
        }
    ).reindex(classes)
)

## 5. Prepare numeric and categorical columns

In [ ]:
numeric_columns = X_train.select_dtypes(include="number").columns
categorical_columns = X_train.select_dtypes(exclude="number").columns

numeric_steps = Pipeline(
    [
        ("fill", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_steps = Pipeline(
    [
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessing = ColumnTransformer(
    [
        ("numeric", numeric_steps, numeric_columns),
        ("categorical", categorical_steps, categorical_columns),
    ]
)

## 6. Create the three models.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(),
}

pipelines = {}
for name, model in models.items():
    pipelines[name] = Pipeline([
        ("preprocessing", clone(preprocessing)),
        ("model", model),
    ])

## 7. Train and evaluate on validation data
- **Accuracy:** fraction of predictions that are correct.
- **Precision:** when the model predicts a class, how often is it right?
- **Recall:** how many actual members of a class does it find?
- **F1:** balances precision and recall.

In [ ]:
results = []

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_val)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(y_val, predictions, average="macro", zero_division=0),
        "Recall": recall_score(y_val, predictions, average="macro", zero_division=0),
        "F1": f1_score(y_val, predictions, average="macro", zero_division=0),
    })

    print(name)
    print(classification_report(y_val, predictions, labels=classes, zero_division=0))

results = pd.DataFrame(results).set_index("Model")
display(results.round(3))

## 8. Check stability with stratified 5-fold cross-validation
Use only the training split. Each fold refits the entire pipeline.
A higher mean F1 is better; a smaller standard deviation means the scores vary
less across these five folds. Stability alone does not imply accuracy.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipeline in pipelines.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_macro")
    results.loc[name, "CV mean"] = scores.mean()
    results.loc[name, "CV std"] = scores.std()

display(results.round(3))

## 9. Choose the best model
Rank by **validation macro F1**. The test set has not been used to choose the
winner. After selection, refit on training + validation data.

In [ ]:
results = results.sort_values("F1", ascending=False)
display(results.round(3))

best_name = results.index[0]
print("Best model:", best_name)

best_pipeline = clone(pipelines[best_name])
best_pipeline.fit(X_train_val, y_train_val)

## 10. Evaluate the winner on the test set

In [ ]:
test_predictions = best_pipeline.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, test_predictions), 3))
print(classification_report(y_test, test_predictions, labels=classes, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=classes, cmap="Blues"
)
plt.title(best_name)
plt.show()

## 11. Inspect classification errors
The largest count below is the most common type of mistake. Inspect a few
rows to understand the errors; do not use the test set to tune the model.

In [ ]:
errors = X_test.copy()
errors["Actual"] = y_test
errors["Predicted"] = test_predictions
errors = errors[errors["Actual"] != errors["Predicted"]]

print("Number of errors:", len(errors))
display(errors[["Actual", "Predicted"]].value_counts())
display(errors.head())

## 12. Show Random Forest feature importance


In [ ]:
forest = pipelines["Random Forest"]
feature_names = forest.named_steps["preprocessing"].get_feature_names_out()
importance = pd.Series(
    forest.named_steps["model"].feature_importances_, index=feature_names
)

importance.nlargest(10).sort_values().plot.barh(figsize=(8, 5))
plt.title("Random Forest: top 10 features")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()

## 13. Save and reload the best pipeline

In [ ]:
joblib.dump(best_pipeline, "best_student_mental_health_pipeline.joblib")
loaded_pipeline = joblib.load("best_student_mental_health_pipeline.joblib")

## 14. Predict for a new student

In [ ]:
new_student = pd.DataFrame([{
    "Age": 21,
    "Gender": "Female",
    "Year_of_Study": 3,
    "Sleep_Hours": 7.0,
    "Study_Hours_Per_Week": 20.0,
    "GPA": 3.1,
    "Social_Media_Hours": 3.0,
    "Screen_Time_Hours": 6.0,
    "Physical_Activity": "Moderate",
    "Financial_Stress": 4.0,
    "Family_Support": 8.0,
    "Attendance_Rate": 90.0,
    "Part_Time_Job": "No",
    "Diet_Quality": "Good",
    "Previous_Mental_Health_History": "No",
}])

print("Predicted risk:", loaded_pipeline.predict(new_student)[0])